In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Axisymmetric B-field of finite-thickness circular loops (class-based, no globals).

- 一様電流密度のワイヤ断面をタイル分割→各タイルをフィラメント環として厳密式(楕円積分K,E)で合成
- 状態はすべてクラスのメンバに保持（グローバルなし）
- 計算/保存/読込/可視化/インタラクティブGUI/ライン出力をメソッドで分離
- tqdmで全ループに進捗、等尺スケール、ベクトル表示ON/OFF・スケール・間引き変更OK
- 計測サマリは print_time_summary()

[最近の追加]
- plot(): width/height、カラーバー範囲(zmin/zmax)
- plot_line_profile(): 指定直線( r=a or z=a )・方向('r' or 'z')・量("|B|","Br","Bz")のXY
- interactive_line_profile(): a/方向/量をGUIで変更＆即時再描画 + 2つの保存ボタン
- sample_field(r, z): 任意(r,z)で Br, Bz を取得（bilinear/nearest）
- compute(): r,z計算領域の手動上書き（r_min, r_max, z_min, z_max）

Requirements: numpy, scipy, plotly, tqdm
Optional: ipywidgets (GUI), pandas + (openpyxl or xlsxwriter) (Excel保存), tkinter (ファイルダイアログ)
"""

from __future__ import annotations
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from contextlib import contextmanager
import time, json
import numpy as np
from tqdm.auto import tqdm
from scipy.special import ellipk, ellipe
import plotly.graph_objects as go
import plotly.figure_factory as ff
# レンダラーは呼び出し側で設定推奨（例：pio.renderers.default = "notebook_connected"）

@dataclass
class AxisymmetricLoopField:
    coil_info: List[Dict[str, float]]                # [{"R":..., "z":..., "dwire":..., "I":...}, ...]
    tiles_per_diameter: int = 11
    pad_ratio_r: float = 0.6
    pad_ratio_z: float = 0.6
    r_floor: float = 1e-12
    mu0: float = 4e-7 * np.pi

    # 計測
    _t0: float = field(default_factory=time.time, init=False)
    _timings: Dict[str, float] = field(default_factory=dict, init=False)

    # 計算結果
    r_vec: Optional[np.ndarray] = field(default=None, init=False)
    z_vec: Optional[np.ndarray] = field(default=None, init=False)
    R: Optional[np.ndarray]     = field(default=None, init=False)
    Z: Optional[np.ndarray]     = field(default=None, init=False)
    Br: Optional[np.ndarray]    = field(default=None, init=False)
    Bz: Optional[np.ndarray]    = field(default=None, init=False)
    Bmag: Optional[np.ndarray]  = field(default=None, init=False)
    filaments_per_coil: Optional[List[List[Dict[str, float]]]] = field(default=None, init=False)

    # ---------- 計測ユーティリティ ----------
    @contextmanager
    def _timer(self, name: str):
        t0 = time.time()
        try:
            yield
        finally:
            self._timings[name] = self._timings.get(name, 0.0) + (time.time() - t0)

    def print_time_summary(self):
        total = time.time() - self._t0
        print("\n===== Timing summary (instance) =====")
        print(f"Total wall time: {total:.3f} s")
        ssum = 0.0
        for k, v in sorted(self._timings.items(), key=lambda kv: kv[1], reverse=True):
            print(f"{k:>28s}: {v:8.3f} s")
            ssum += v
        print(f"Sum of function times: {ssum:.3f} s")
        print("====================================\n")

    # ---------- 物理コア：単一フィラメント環 ----------
    def _B_loop_rz(self, r: np.ndarray, z: np.ndarray, a: float, z0: float, I: float,
                   r_eps: float = 1e-12, m_eps: float = 1e-12, q_eps: float = 1e-18
                   ) -> Tuple[np.ndarray, np.ndarray]:
        with self._timer("_B_loop_rz"):
            r = np.asarray(r, float); z = np.asarray(z, float)
            zz = z - z0
            Br = np.zeros_like(r); Bz = np.zeros_like(r)
            on_axis = (np.abs(r) < r_eps)

            if np.any(~on_axis):
                rr = r[~on_axis]; zzg = zz[~on_axis]
                denom = (a + rr)**2 + zzg**2
                m = np.clip(4*a*rr/denom, 0.0, 1.0 - m_eps)
                K = ellipk(m); E = ellipe(m)
                S = np.sqrt(denom)
                Q = (a - rr)**2 + zzg**2 + q_eps

                Br_loc = (self.mu0*I*zzg)/(2*np.pi*rr*S) * (-K + ((a**2 + rr**2 + zzg**2)/Q)*E)
                Bz_loc = (self.mu0*I)/(2*np.pi*S) * ( K + ((a**2 - rr**2 - zzg**2)/Q)*E )
                Br[~on_axis] = Br_loc; Bz[~on_axis] = Bz_loc

            if np.any(on_axis):
                zz0 = zz[on_axis]
                Bz_axis = self.mu0*I*a*a / (2.0)*(a*a + zz0*zz0)**(-1.5)
                Br[on_axis] = 0.0; Bz[on_axis] = Bz_axis
            return Br, Bz

    # ---------- 断面→フィラメント群 ----------
    def _discretize_wire_as_filaments(self, R0: float, z0: float, D: float, I_total: float
                                      ) -> List[Dict[str, float]]:
        with self._timer("_discretize_wire_as_filaments"):
            if D <= 0:
                return [{"a": max(R0, self.r_floor), "z0": z0, "I": I_total}]

            N = max(3, int(self.tiles_per_diameter))
            dr = dz = D / N
            r_min = R0 - D/2; r_max = R0 + D/2
            z_min = z0 - D/2; z_max = z0 + D/2
            r_centers = np.linspace(r_min + dr/2, r_max - dr/2, N)
            z_centers = np.linspace(z_min + dz/2, z_max - dz/2, N)
            rr, zz = np.meshgrid(r_centers, z_centers)
            mask = (rr - R0)**2 + (zz - z0)**2 <= (D/2)**2
            rr_sel = rr[mask]; zz_sel = zz[mask]

            if rr_sel.size == 0:
                return [{"a": max(R0, self.r_floor), "z0": z0, "I": I_total}]

            dA = dr * dz
            A_eff = rr_sel.size * dA
            J = I_total / A_eff

            filaments: List[Dict[str, float]] = []
            for a_i, z_i in tqdm(list(zip(rr_sel, zz_sel)), total=rr_sel.size,
                                 desc="discretize: assign currents"):
                a_i = max(float(a_i), self.r_floor)
                filaments.append({"a": a_i, "z0": float(z_i), "I": float(J*dA)})

            I_sum = sum(f["I"] for f in filaments)
            if I_sum > 0:
                scale = I_total / I_sum
                for f in tqdm(filaments, desc="discretize: normalize I"):
                    f["I"] *= scale
            return filaments

    # ---------- ドメイン/グリッド ----------
    def _auto_domain_from_coils(self) -> Tuple[float, float, float]:
        with self._timer("_auto_domain_from_coils"):
            Rmax = max(c["R"] + c.get("dwire", 0.0)/2 for c in self.coil_info)
            r_max = (1.0 + self.pad_ratio_r) * Rmax
            z_min = min(c["z"] - c.get("dwire", 0.0)/2 for c in self.coil_info)
            z_max = max(c["z"] + c.get("dwire", 0.0)/2 for c in self.coil_info)
            z_span = z_max - z_min if z_max > z_min else max(1e-3, self.coil_info[0].get("dwire", 1e-3))
            z_min -= self.pad_ratio_z * z_span
            z_max += self.pad_ratio_z * z_span
            return r_max, z_min, z_max

    def _make_rz_grid(self, r_min: float, r_max: float, z_min: float, z_max: float, Nr: int, Nz: int
                      ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        with self._timer("_make_rz_grid"):
            r_vec = np.linspace(r_min, r_max, Nr)
            z_vec = np.linspace(z_min, z_max, Nz)
            R, Z = np.meshgrid(r_vec, z_vec)
            return r_vec, z_vec, R, Z

    # ---------- 計算本体（領域の上書き指定に対応） ----------
    def compute(self,
                Nr: int = 201, Nz: int = 201,
                r_max: Optional[float] = None,
                z_min: Optional[float] = None,
                z_max: Optional[float] = None,
                r_min: float = 0.0):
        """
        フィールド計算。領域は既定で自動推定だが、引数で上書き可能。
          - r ∈ [r_min, r_max]（既定 r_min=0）
          - z ∈ [z_min, z_max]
        """
        with self._timer("compute"):
            auto_r_max, auto_z_min, auto_z_max = self._auto_domain_from_coils()

            RMAX = float(auto_r_max if r_max is None else r_max)
            ZMIN = float(auto_z_min if z_min is None else z_min)
            ZMAX = float(auto_z_max if z_max is None else z_max)

            if RMAX <= r_min:
                raise ValueError(f"r_max({RMAX}) must be > r_min({r_min}).")
            if ZMAX <= ZMIN:
                raise ValueError(f"z_max({ZMAX}) must be > z_min({ZMIN}).")

            self.r_vec, self.z_vec, self.R, self.Z = self._make_rz_grid(r_min, RMAX, ZMIN, ZMAX, Nr, Nz)

            Br_tot = np.zeros_like(self.R); Bz_tot = np.zeros_like(self.Z)
            self.filaments_per_coil = []

            for c in tqdm(self.coil_info, desc="compute: coils"):
                R0 = float(c["R"]); z0 = float(c["z"]); D = float(c["dwire"]); I_tot = float(c.get("I", 1.0))
                fils = self._discretize_wire_as_filaments(R0, z0, D, I_tot)
                self.filaments_per_coil.append(fils)
                for f in tqdm(fils, desc="compute: filaments"):
                    Br_i, Bz_i = self._B_loop_rz(self.R, self.Z, a=f["a"], z0=f["z0"], I=f["I"])
                    Br_tot += Br_i; Bz_tot += Bz_i

            self.Br, self.Bz = Br_tot, Bz_tot
            self.Bmag = np.sqrt(Br_tot**2 + Bz_tot**2)

    # ---------- 保存/読込 ----------
    def save(self, path: str):
        with self._timer("save"):
            meta = {
                "coil_info": self.coil_info,
                "tiles_per_diameter": self.tiles_per_diameter,
                "pad_ratio_r": self.pad_ratio_r,
                "pad_ratio_z": self.pad_ratio_z,
                "r_floor": self.r_floor,
                "mu0": self.mu0,
            }
            np.savez_compressed(
                path,
                r_vec=self.r_vec, z_vec=self.z_vec, R=self.R, Z=self.Z,
                Br=self.Br, Bz=self.Bz, Bmag=self.Bmag,
                meta_json=json.dumps(meta),
            )

    @classmethod
    def load(cls, path: str) -> "AxisymmetricLoopField":
        with np.load(path, allow_pickle=False) as npz:
            meta = json.loads(str(npz["meta_json"]))
            obj = cls(**meta)
            obj.r_vec = npz["r_vec"]; obj.z_vec = npz["z_vec"]
            obj.R = npz["R"]; obj.Z = npz["Z"]
            obj.Br = npz["Br"]; obj.Bz = npz["Bz"]; obj.Bmag = npz["Bmag"]
            return obj

    # ---------- 可視化（幅・高さ・カラーバー範囲） ----------
    def plot(self,
             component: str = "Bmag",              # "Bmag" or "Bz"
             heatmap_colorscale: str = "Viridis",
             show_vectors: bool = True,
             quiver_stride: int = 6,
             quiver_maxlen_frac: float = 0.12,
             quiver_scale: Optional[float] = None,
             title: Optional[str] = None,
             show: bool = True,
             width: Optional[int] = None,
             height: Optional[int] = None,
             zmin: Optional[float] = None,
             zmax: Optional[float] = None
             ) -> go.Figure:
        with self._timer("plot"):
            assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
            assert self.Br is not None and self.Bz is not None, "No field arrays."

            Zplot = self.Bmag if component.lower() != "bz" else self.Bz
            z_title = "|B| [T]" if component.lower() != "bz" else "Bz [T]"

            fig = go.Figure()
            fig.add_trace(go.Heatmap(
                x=self.r_vec, y=self.z_vec, z=Zplot,
                colorscale=heatmap_colorscale,
                colorbar=dict(title=z_title),
                zmin=zmin, zmax=zmax
            ))

            if show_vectors:
                s = max(1, int(quiver_stride))
                r_sub = self.r_vec[::s]; z_sub = self.z_vec[::s]
                R_sub, Z_sub = np.meshgrid(r_sub, z_sub)
                U = self.Br[::s, ::s]; V = self.Bz[::s, ::s]

                if quiver_scale is None:
                    B_sub = np.sqrt(U*U + V*V)
                    Bref = np.percentile(B_sub[B_sub > 0], 99) if np.any(B_sub > 0) else 1.0
                    r_span = float(self.r_vec[-1] - self.r_vec[0])
                    z_span = float(self.z_vec[-1] - self.z_vec[0])
                    base_len = quiver_maxlen_frac * min(r_span, z_span)
                    scale = base_len / (Bref + 1e-30)
                else:
                    scale = float(quiver_scale)

                qfig = ff.create_quiver(R_sub.flatten(), Z_sub.flatten(),
                                        U.flatten(), V.flatten(),
                                        scale=scale, arrow_scale=0.25,
                                        name="B (Br,Bz)", line_width=1)
                for tr in tqdm(qfig.data, desc="plot: add quiver traces"):
                    fig.add_trace(tr)

            # ワイヤ断面輪郭
            shapes = []
            for c in tqdm(self.coil_info, desc="plot: add wire outlines"):
                R0 = float(c["R"]); z0 = float(c["z"]); D = float(c.get("dwire", 0.0))
                if D > 0:
                    shapes.append(dict(
                        type="circle", xref="x", yref="y", layer="above",
                        x0=R0 - D/2, x1=R0 + D/2, y0=z0 - D/2, y1=z0 + D/2,
                        line=dict(width=2, color="black")
                    ))
            fig.update_layout(
                title=title or f"Magnetic field on (r,z) — Heatmap: {z_title}" + (" + vectors" if show_vectors else ""),
                xaxis_title="r [m]", yaxis_title="z [m]",
                template="plotly_white", shapes=shapes,
                width=width, height=height
            )
            # 等尺
            fig.update_yaxes(scaleanchor="x", scaleratio=1)
            if show:
                fig.show()
            return fig

    # ---------- 直線上プロファイル ----------
    def plot_line_profile(self,
                          a: float,
                          direction: str = "z",
                          quantity: str = "|B|",
                          title: Optional[str] = None,
                          show: bool = True,
                          width: Optional[int] = None,
                          height: Optional[int] = None
                          ) -> go.Figure:
        with self._timer("plot_line_profile"):
            assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
            assert self.Br is not None and self.Bz is not None and self.Bmag is not None, "No field arrays."

            qkey = quantity.lower()
            if qkey == "br":
                Q = self.Br; y_label = "Br [T]"
            elif qkey == "bz":
                Q = self.Bz; y_label = "Bz [T]"
            else:
                Q = self.Bmag; y_label = "|B| [T]"

            direction = direction.lower()
            if direction == "z":
                r_idx = int(np.argmin(np.abs(self.r_vec - a)))
                x = self.z_vec; y = Q[:, r_idx]
                x_label = "z [m]"; used = self.r_vec[r_idx]
                ttl = title or f"Line profile along z at r≈{used:.6g} m ({quantity})"
            elif direction == "r":
                z_idx = int(np.argmin(np.abs(self.z_vec - a)))
                x = self.r_vec; y = Q[z_idx, :]
                x_label = "r [m]"; used = self.z_vec[z_idx]
                ttl = title or f"Line profile along r at z≈{used:.6g} m ({quantity})"
            else:
                raise ValueError("direction must be 'z' or 'r'.")

            fig = go.Figure()
            fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=quantity))
            fig.update_layout(
                title=ttl,
                xaxis_title=x_label,
                yaxis_title=y_label,
                template="plotly_white",
                width=width, height=height
            )
            if show:
                fig.show()
            return fig

    # ADDED 2025-09-22  
    def save_line_profile(self,
                        path: str,
                        a: float,
                        direction: str = "z",
                        quantity: str = "|B|") -> None:
        """
        現在のフィールド配列から、指定直線（r=a または z=a）のラインプロファイルを計算し、
        Excel(.xlsx)に保存する。ファイルダイアログは使わない。
        - direction='z': r=a に最も近い列で z を横軸に保存
        - direction='r': z=a に最も近い行で r を横軸に保存
        - quantity: "|B|" or "Br" or "Bz"
        """
        with self._timer("save_line_profile"):
            import os
            try:
                import pandas as pd
            except Exception:
                raise RuntimeError("pandas が必要です。`pip install pandas openpyxl` を実行してください。")

            assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
            assert self.Br is not None and self.Bz is not None and self.Bmag is not None, "No field arrays."

            qkey = quantity.lower()
            if qkey == "br":
                Q = self.Br; y_label = "Br [T]"
            elif qkey == "bz":
                Q = self.Bz; y_label = "Bz [T]"
            else:
                Q = self.Bmag; y_label = "|B| [T]"

            direction = direction.lower()
            if direction == "z":
                r_idx = int(np.argmin(np.abs(self.r_vec - a)))
                x = self.z_vec; y = Q[:, r_idx]
                x_label = "z [m]"
            elif direction == "r":
                z_idx = int(np.argmin(np.abs(self.z_vec - a)))
                x = self.r_vec; y = Q[z_idx, :]
                x_label = "r [m]"
            else:
                raise ValueError("direction must be 'z' or 'r'.")

            if not path.lower().endswith(".xlsx"):
                path += ".xlsx"
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

            # openpyxlがあれば優先、無ければxlsxwriter
            engine = "openpyxl"
            try:
                import openpyxl  # noqa
            except Exception:
                engine = "xlsxwriter"

            df = pd.DataFrame({x_label: x, y_label: y})
            with pd.ExcelWriter(path, engine=engine) as writer:
                df.to_excel(writer, sheet_name="LineProfile", index=False)

    def save_maps_to_excel(self, path: str) -> None:
        """
        Br, Bz の2シートをもつExcelを出力する。ダイアログは使わない。
        - シートBr/Bzそれぞれで:
            A列の2行目以降に z
            1行目のB列以降に r
            B2セル以降に値行列
        """
        with self._timer("save_maps_to_excel"):
            import os
            try:
                import xlsxwriter
            except Exception:
                raise RuntimeError("`pip install xlsxwriter` を実行してください。")

            assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
            assert self.Br is not None and self.Bz is not None, "No field arrays."

            if not path.lower().endswith(".xlsx"):
                path += ".xlsx"
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

            Nz, Nr = self.Br.shape
            wb = xlsxwriter.Workbook(path)
            try:
                # --- Br ---
                ws = wb.add_worksheet("Br")
                for j, rv in enumerate(self.r_vec):
                    ws.write(0, 1 + j, float(rv))      # 1行目B列以降に r
                for i, zv in enumerate(self.z_vec):
                    ws.write(1 + i, 0, float(zv))      # A列2行目以降に z
                for i in range(Nz):
                    ws.write_row(1 + i, 1, self.Br[i, :].tolist())  # B2以降に行単位で書き込み

                # --- Bz ---
                ws2 = wb.add_worksheet("Bz")
                for j, rv in enumerate(self.r_vec):
                    ws2.write(0, 1 + j, float(rv))
                for i, zv in enumerate(self.z_vec):
                    ws2.write(1 + i, 0, float(zv))
                for i in range(Nz):
                    ws2.write_row(1 + i, 1, self.Bz[i, :].tolist())
            finally:
                wb.close()

    # ################# ADDED 2025-09-22  


    def interactive_line_profile(self,
                                a_init: float = 0.0,
                                direction_init: str = "z",
                                quantity_init: str = "|B|",
                                width: Optional[int] = None,
                                height: Optional[int] = None,
                                profile_save_path: Optional[str] = None,
                                maps_save_path: Optional[str] = None):
        """
        Jupyter上のGUIで a/方向/量 を変更→即時再描画。
        保存ボタンはダイアログを出さず、引数で渡された保存先パスに即保存します。
        - profile_save_path: 現在のラインプロファイルを保存する .xlsx パス
        - maps_save_path   : Br/Bz マップを保存する .xlsx パス
        """
        with self._timer("interactive_line_profile"):
            try:
                import ipywidgets as widgets
                from IPython.display import display, clear_output
            except Exception as e:
                raise RuntimeError("ipywidgets が必要です。`pip install ipywidgets` を実行してください。") from e

            a_box = widgets.FloatText(value=float(a_init), description="a =", step=0.001, layout=widgets.Layout(width="200px"))
            dir_dd = widgets.Dropdown(options=[("r=a の線に沿って (→ r)", "r"),
                                            ("z=a の線に沿って (→ z)", "z")],
                                    value=direction_init, description="方向", layout=widgets.Layout(width="320px"))
            qty_dd = widgets.Dropdown(options=["|B|", "Br", "Bz"],
                                    value=quantity_init, description="量", layout=widgets.Layout(width="200px"))

            btn_save_profile = widgets.Button(description="現在のラインプロファイルをExcelに保存", button_style="primary")
            btn_save_maps    = widgets.Button(description="Br/Bz マップをExcelに保存")

            out_plot = widgets.Output()
            out_msg  = widgets.Output()

            _state = {"x": None, "y": None, "x_label": "", "y_label": "", "title": "",
                    "a": float(a_init), "dir": direction_init, "qty": quantity_init}

            def _update(_=None):
                with out_plot:
                    clear_output(wait=True)
                    q = qty_dd.value
                    d = dir_dd.value
                    a = float(a_box.value)

                    # 量の選択
                    qkey = q.lower()
                    if qkey == "br":
                        Q = self.Br; y_label = "Br [T]"
                    elif qkey == "bz":
                        Q = self.Bz; y_label = "Bz [T]"
                    else:
                        Q = self.Bmag; y_label = "|B| [T]"

                    if d == "z":
                        r_idx = int(np.argmin(np.abs(self.r_vec - a)))
                        x = self.z_vec; y = Q[:, r_idx]
                        x_label = "z [m]"; used = self.r_vec[r_idx]
                        ttl = f"Line profile along z at r≈{used:.6g} m ({q})"
                    else:
                        z_idx = int(np.argmin(np.abs(self.z_vec - a)))
                        x = self.r_vec; y = Q[z_idx, :]
                        x_label = "r [m]"; used = self.z_vec[z_idx]
                        ttl = f"Line profile along r at z≈{used:.6g} m ({q})"

                    _state.update({"x": x, "y": y, "x_label": x_label, "y_label": y_label,
                                "title": ttl, "a": a, "dir": d, "qty": q})

                    fig = go.Figure()
                    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=q))
                    fig.update_layout(title=ttl, xaxis_title=x_label, yaxis_title=y_label,
                                    template="plotly_white", width=width, height=height)
                    fig.show()

            def _save_profile_clicked(_):
                with out_msg:
                    out_msg.clear_output(wait=True)
                    if profile_save_path is None:
                        print("profile_save_path が未指定です。interactive_line_profile(..., profile_save_path='.../line_profile.xlsx') を渡してください。")
                        return
                    try:
                        # 現在表示中の設定（a, dir, qty）で保存
                        self.save_line_profile(profile_save_path, a=_state["a"], direction=_state["dir"], quantity=_state["qty"])
                        print(f"保存しました: {profile_save_path}")
                    except Exception as e:
                        print(f"保存に失敗しました: {e}")

            def _save_maps_clicked(_):
                with out_msg:
                    out_msg.clear_output(wait=True)
                    if maps_save_path is None:
                        print("maps_save_path が未指定です。interactive_line_profile(..., maps_save_path='.../Br_Bz_maps.xlsx') を渡してください。")
                        return
                    try:
                        self.save_maps_to_excel(maps_save_path)
                        print(f"保存しました: {maps_save_path}")
                    except Exception as e:
                        print(f"保存に失敗しました: {e}")

            a_box.observe(_update, names="value")
            dir_dd.observe(_update, names="value")
            qty_dd.observe(_update, names="value")
            btn_save_profile.on_click(_save_profile_clicked)
            btn_save_maps.on_click(_save_maps_clicked)

            _update()

            ui = widgets.VBox([
                widgets.HBox([a_box, dir_dd, qty_dd]),
                out_plot,
                widgets.HBox([btn_save_profile, btn_save_maps]),
                out_msg
            ])
            display(ui)

    # ---------- 任意点サンプリング（Br, Bz を返す） ----------
    def sample_field(self, r: float, z: float, method: str = "bilinear", clip: bool = True) -> Tuple[float, float]:
        """
        指定座標 (r,z) の Br, Bz を返す。
          method: "bilinear"（双一次補間） or "nearest"（最近傍）
          clip  : 範囲外は境界にクリップ（Falseなら範囲外で ValueError）
        """
        assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
        assert self.Br is not None and self.Bz is not None, "No field arrays."

        r0, r1 = self.r_vec[0], self.r_vec[-1]
        z0, z1 = self.z_vec[0], self.z_vec[-1]
        rr = float(np.clip(r, r0, r1)) if clip else float(r)
        zz = float(np.clip(z, z0, z1)) if clip else float(z)
        if not (r0 <= rr <= r1 and z0 <= zz <= z1):
            raise ValueError(f"(r,z)=({r},{z}) is out of grid range: r∈[{r0},{r1}], z∈[{z0},{z1}]")

        if method.lower() == "nearest":
            i = int(np.argmin(np.abs(self.z_vec - zz)))
            j = int(np.argmin(np.abs(self.r_vec - rr)))
            return float(self.Br[i, j]), float(self.Bz[i, j])

        # bilinear
        i1 = int(np.searchsorted(self.z_vec, zz, side="left"))
        j1 = int(np.searchsorted(self.r_vec, rr, side="left"))
        i0 = max(0, min(i1 - 1, len(self.z_vec) - 2))
        j0 = max(0, min(j1 - 1, len(self.r_vec) - 2))
        zL, zU = self.z_vec[i0], self.z_vec[i0 + 1]
        rL, rU = self.r_vec[j0], self.r_vec[j0 + 1]
        tz = 0.0 if zU == zL else (zz - zL) / (zU - zL)
        tr = 0.0 if rU == rL else (rr - rL) / (rU - rL)

        def _bilin(Q: np.ndarray) -> float:
            q11 = Q[i0, j0]; q12 = Q[i0, j0 + 1]
            q21 = Q[i0 + 1, j0]; q22 = Q[i0 + 1, j0 + 1]
            return float(
                (1 - tz) * ((1 - tr) * q11 + tr * q12) +
                  tz  * ((1 - tr) * q21 + tr * q22)
            )

        return _bilin(self.Br), _bilin(self.Bz)


In [ ]:
# -------------------- 使い方サンプル --------------------
lf = AxisymmetricLoopField(
    coil_info=[
        {"R": 0.10, "z": 0.0, "dwire": 0.003, "I": 1.0},
        # {"R": 0.10, "z": +0.05, "dwire": 0.003, "I": 10.0},
    ],
    tiles_per_diameter=11,
    pad_ratio_r=0.5, pad_ratio_z=0.5,
    r_floor=1e-12
)

lf.compute(Nr=601, Nz=601, r_min=0.0, r_max=0.25, z_min=-0.15, z_max=0.15)


In [ ]:
# ヒートマップ（サイズ/カラーバー範囲を指定）
lf.plot(component="Bmag", heatmap_colorscale="Viridis",
        show_vectors=True, quiver_stride=6, quiver_maxlen_frac=0.12,
        quiver_scale=None, title="Class-based |B|",
        width=900, height=700, zmin=None, zmax=None, show=True)

# 2) 任意点の Br, Bz（双一次補間）
Br_val, Bz_val = lf.sample_field(r=0.07, z=0.01, method="bilinear")
print(Br_val, Bz_val)

# 直線上プロファイル（静的）
# lf.plot_line_profile(a=0.05, direction="z", quantity="|B|",
#                       width=800, height=400, show=True)

# 3) ラインプロファイルをコードから保存（ダイアログなし）
lf.save_line_profile(path="./outputs/line_profile_z_at_r=0.05.xlsx",
                     a=0.05, direction="z", quantity="|B|")

# 4) Br/Bz マップをコードから保存（ダイアログなし）
lf.save_maps_to_excel(path="./outputs/Br_Bz_maps.xlsx")

# 5) JupyterのインタラクティブGUI（保存パスを引数で指定）
# import plotly.io as pio; pio.renderers.default = "notebook_connected"
lf.interactive_line_profile(a_init=0.05, direction_init="z", quantity_init="|B|",
                            width=800, height=400)

lf.print_time_summary()
  

# 多数導体テスト

In [ ]:
# -------------------- 使い方サンプル --------------------
R_inner = 0.10
N_per_layer = 10
N_layer = 5
d_wire = 0.003
I_coil = 1.0
coil_info_list = []
for i in range(N_layer):
    z_i = (i - (N_layer - 1)/2) * (d_wire + 0.001)
    for j in range(N_per_layer):
        R_j = R_inner + j * (d_wire + 0.001)
        coil_info_list.append({"R": R_j, "z": z_i, "dwire": d_wire, "I": I_coil})

lf = AxisymmetricLoopField(
    coil_info=coil_info_list,
    tiles_per_diameter=11,
    pad_ratio_r=0.5, pad_ratio_z=0.5,
    r_floor=1e-12
)

lf.compute(Nr=801, Nz=801)


In [ ]:
# ヒートマップ（サイズ/カラーバー範囲を指定）
lf.plot(component="Bmag", heatmap_colorscale="Viridis",
        show_vectors=True, quiver_stride=6, quiver_maxlen_frac=0.12,
        quiver_scale=None, title="Class-based |B|",
        width=900, height=700, zmin=None, zmax=None, show=True)

# 2) 任意点の Br, Bz（双一次補間）
Br_val, Bz_val = lf.sample_field(r=0.07, z=0.01, method="bilinear")
print(Br_val, Bz_val)

# 直線上プロファイル（静的）
# lf.plot_line_profile(a=0.05, direction="z", quantity="|B|",
#                       width=800, height=400, show=True)

# 3) ラインプロファイルをコードから保存（ダイアログなし）
lf.save_line_profile(path="./outputs/line_profile_z_at_r=0.05.xlsx",
                     a=0.05, direction="z", quantity="|B|")

# 4) Br/Bz マップをコードから保存（ダイアログなし）
lf.save_maps_to_excel(path="./outputs/Br_Bz_maps.xlsx")

# 5) JupyterのインタラクティブGUI（保存パスを引数で指定）
# import plotly.io as pio; pio.renderers.default = "notebook_connected"
lf.interactive_line_profile(a_init=0.05, direction_init="z", quantity_init="|B|",
                            width=800, height=400)

lf.print_time_summary()
  

In [ ]:
# 依存: numpy, scipy, plotly, tqdm
# オプション: ipywidgets, pandas, openpyxl/xlsxwriter（保存系）

from __future__ import annotations
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, DefaultDict
from contextlib import contextmanager
import time, json, math
import numpy as np
from tqdm.auto import tqdm
from scipy.special import ellipk, ellipe
import plotly.graph_objects as go
import plotly.figure_factory as ff
from collections import defaultdict

@dataclass
class AxisymmetricLoopField:
    coil_info: List[Dict[str, float]]                # [{"R":..., "z":..., "dwire":..., "I":...}, ...]
    tiles_per_diameter: int = 11
    pad_ratio_r: float = 0.6
    pad_ratio_z: float = 0.6
    r_floor: float = 1e-12
    mu0: float = 4e-7 * np.pi

    # 計測
    _t0: float = field(default_factory=time.time, init=False)
    _timings: Dict[str, float] = field(default_factory=dict, init=False)

    # 計算結果
    r_vec: Optional[np.ndarray] = field(default=None, init=False)
    z_vec: Optional[np.ndarray] = field(default=None, init=False)
    R: Optional[np.ndarray]     = field(default=None, init=False)
    Z: Optional[np.ndarray]     = field(default=None, init=False)
    Br: Optional[np.ndarray]    = field(default=None, init=False)
    Bz: Optional[np.ndarray]    = field(default=None, init=False)
    Bmag: Optional[np.ndarray]  = field(default=None, init=False)
    filaments_per_coil: Optional[List[List[Dict[str, float]]]] = field(default=None, init=False)

    # ---------- 計測ユーティリティ ----------
    @contextmanager
    def _timer(self, name: str):
        t0 = time.time()
        try:
            yield
        finally:
            self._timings[name] = self._timings.get(name, 0.0) + (time.time() - t0)

    def print_time_summary(self):
        total = time.time() - self._t0
        print("\n===== Timing summary (instance) =====")
        print(f"Total wall time: {total:.3f} s")
        ssum = 0.0
        for k, v in sorted(self._timings.items(), key=lambda kv: kv[1], reverse=True):
            print(f"{k:>28s}: {v:8.3f} s")
            ssum += v
        print(f"Sum of function times: {ssum:.3f} s")
        print("====================================\n")

    # ---------- 物理コア：単一フィラメント環 ----------
    def _B_loop_rz(self, r: np.ndarray, z: np.ndarray, a: float, z0: float, I: float,
                   r_eps: float = 1e-12, m_eps: float = 1e-12, q_eps: float = 1e-18
                   ) -> Tuple[np.ndarray, np.ndarray]:
        with self._timer("_B_loop_rz"):
            r = np.asarray(r, float); z = np.asarray(z, float)
            zz = z - z0
            Br = np.zeros_like(r); Bz = np.zeros_like(r)
            on_axis = (np.abs(r) < r_eps)

            if np.any(~on_axis):
                rr = r[~on_axis]; zzg = zz[~on_axis]
                denom = (a + rr)**2 + zzg**2
                m = np.clip(4*a*rr/denom, 0.0, 1.0 - m_eps)
                K = ellipk(m); E = ellipe(m)
                S = np.sqrt(denom)
                Q = (a - rr)**2 + zzg**2 + q_eps

                Br_loc = (self.mu0*I*zzg)/(2*np.pi*rr*S) * (-K + ((a**2 + rr**2 + zzg**2)/Q)*E)
                Bz_loc = (self.mu0*I)/(2*np.pi*S) * ( K + ((a**2 - rr**2 - zzg**2)/Q)*E )
                Br[~on_axis] = Br_loc; Bz[~on_axis] = Bz_loc

            if np.any(on_axis):
                zz0 = zz[on_axis]
                # 修正: 軸上式 (mu0*I*a^2) / (2 (a^2 + z^2)^(3/2))
                Bz_axis = self.mu0*I*a*a / (2.0 * (a*a + zz0*zz0)**1.5)
                Br[on_axis] = 0.0; Bz[on_axis] = Bz_axis
            return Br, Bz

    # ---------- 断面→フィラメント群 ----------
    def _discretize_wire_as_filaments(self, R0: float, z0: float, D: float, I_total: float
                                      ) -> List[Dict[str, float]]:
        with self._timer("_discretize_wire_as_filaments"):
            if D <= 0:
                return [{"a": max(R0, self.r_floor), "z0": z0, "I": I_total}]

            N = max(3, int(self.tiles_per_diameter))
            dr = dz = D / N
            r_min = R0 - D/2; r_max = R0 + D/2
            z_min = z0 - D/2; z_max = z0 + D/2
            r_centers = np.linspace(r_min + dr/2, r_max - dr/2, N)
            z_centers = np.linspace(z_min + dz/2, z_max - dz/2, N)
            rr, zz = np.meshgrid(r_centers, z_centers)
            mask = (rr - R0)**2 + (zz - z0)**2 <= (D/2)**2
            rr_sel = rr[mask]; zz_sel = zz[mask]

            if rr_sel.size == 0:
                return [{"a": max(R0, self.r_floor), "z0": z0, "I": I_total}]

            dA = dr * dz
            A_eff = rr_sel.size * dA
            J = I_total / A_eff

            filaments: List[Dict[str, float]] = []
            for a_i, z_i in tqdm(list(zip(rr_sel, zz_sel)), total=rr_sel.size,
                                 desc="discretize: assign currents"):
                a_i = max(float(a_i), self.r_floor)
                filaments.append({"a": a_i, "z0": float(z_i), "I": float(J*dA)})

            I_sum = sum(f["I"] for f in filaments)
            if I_sum > 0:
                scale = I_total / I_sum
                for f in tqdm(filaments, desc="discretize: normalize I"):
                    f["I"] *= scale
            return filaments

    # ---------- ドメイン/グリッド ----------
    def _auto_domain_from_coils(self) -> Tuple[float, float, float]:
        with self._timer("_auto_domain_from_coils"):
            Rmax = max(c["R"] + c.get("dwire", 0.0)/2 for c in self.coil_info)
            r_max = (1.0 + self.pad_ratio_r) * Rmax
            z_min = min(c["z"] - c.get("dwire", 0.0)/2 for c in self.coil_info)
            z_max = max(c["z"] + c.get("dwire", 0.0)/2 for c in self.coil_info)
            z_span = z_max - z_min if z_max > z_min else max(1e-3, self.coil_info[0].get("dwire", 1e-3))
            z_min -= self.pad_ratio_z * z_span
            z_max += self.pad_ratio_z * z_span
            return r_max, z_min, z_max

    def _make_rz_grid(self, r_min: float, r_max: float, z_min: float, z_max: float, Nr: int, Nz: int
                      ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        with self._timer("_make_rz_grid"):
            r_vec = np.linspace(r_min, r_max, Nr)
            z_vec = np.linspace(z_min, z_max, Nz)
            R, Z = np.meshgrid(r_vec, z_vec)
            return r_vec, z_vec, R, Z

    # ---------- 計算本体（通常） or （高速化：zシフト再利用） ----------
    def compute(self,
                Nr: int = 201, Nz: int = 201,
                r_max: Optional[float] = None,
                z_min: Optional[float] = None,
                z_max: Optional[float] = None,
                r_min: float = 0.0,
                use_shift_accel: bool = False,
                base_pad_frac: float = 0.05):
        """
        フィールド計算。領域は既定で自動推定だが、引数で上書き可能。
        use_shift_accel=True で (R,dwire) ごとの基準場を z シフト再利用して高速化。
        base_pad_frac: 基準場 z 範囲の余裕（解析領域スパンの何倍か）。デフォルト5%。
        """
        if use_shift_accel:
            return self._compute_via_shift_groups(Nr, Nz, r_max, z_min, z_max, r_min, base_pad_frac)

        with self._timer("compute"):
            auto_r_max, auto_z_min, auto_z_max = self._auto_domain_from_coils()
            RMAX = float(auto_r_max if r_max is None else r_max)
            ZMIN = float(auto_z_min if z_min is None else z_min)
            ZMAX = float(auto_z_max if z_max is None else z_max)
            if RMAX <= r_min:
                raise ValueError(f"r_max({RMAX}) must be > r_min({r_min}).")
            if ZMAX <= ZMIN:
                raise ValueError(f"z_max({ZMAX}) must be > z_min({ZMIN}).")

            self.r_vec, self.z_vec, self.R, self.Z = self._make_rz_grid(r_min, RMAX, ZMIN, ZMAX, Nr, Nz)
            Br_tot = np.zeros_like(self.R); Bz_tot = np.zeros_like(self.Z)
            self.filaments_per_coil = []

            for c in tqdm(self.coil_info, desc="compute: coils"):
                R0 = float(c["R"]); z0 = float(c["z"]); D = float(c["dwire"]); I_tot = float(c.get("I", 1.0))
                fils = self._discretize_wire_as_filaments(R0, z0, D, I_tot)
                self.filaments_per_coil.append(fils)
                for f in tqdm(fils, desc="compute: filaments"):
                    Br_i, Bz_i = self._B_loop_rz(self.R, self.Z, a=f["a"], z0=f["z0"], I=f["I"])
                    Br_tot += Br_i; Bz_tot += Bz_i

            self.Br, self.Bz = Br_tot, Bz_tot
            self.Bmag = np.sqrt(Br_tot**2 + Bz_tot**2)

    # ---------- 高速版：zシフト再利用 ----------
    def _compute_via_shift_groups(self,
                                  Nr: int, Nz: int,
                                  r_max: Optional[float], z_min: Optional[float], z_max: Optional[float],
                                  r_min: float, base_pad_frac: float):
        with self._timer("compute_fast_shift"):
            # 1) 解析領域の決定
            auto_r_max, auto_z_min, auto_z_max = self._auto_domain_from_coils()
            RMAX = float(auto_r_max if r_max is None else r_max)
            ZMIN = float(auto_z_min if z_min is None else z_min)
            ZMAX = float(auto_z_max if z_max is None else z_max)
            if RMAX <= r_min:
                raise ValueError(f"r_max({RMAX}) must be > r_min({r_min}).")
            if ZMAX <= ZMIN:
                raise ValueError(f"z_max({ZMAX}) must be > z_min({ZMIN}).")
            self.r_vec, self.z_vec, self.R, self.Z = self._make_rz_grid(r_min, RMAX, ZMIN, ZMAX, Nr, Nz)
            dz = (self.z_vec[-1] - self.z_vec[0]) / (len(self.z_vec) - 1)

            Br_tot = np.zeros_like(self.R); Bz_tot = np.zeros_like(self.Z)

            # 2) グルーピング (R, dwire) 毎
            groups: DefaultDict[Tuple[float, float], List[Dict[str, float]]] = defaultdict(list)
            for c in self.coil_info:
                key = (float(c["R"]), float(c["dwire"]))
                groups[key].append(c)

            # 3) 各グループごとに基準場を作成し、zシフトで合成
            for (R0, D0), items in tqdm(groups.items(), desc="fast: groups"):
                z_list = [float(c["z"]) for c in items]
                I_list = [float(c.get("I", 1.0)) for c in items]
                z_min_shift = ZMIN - max(z_list)
                z_max_shift = ZMAX - min(z_list)
                span = ZMAX - ZMIN
                pad = base_pad_frac * span
                base_z_min = z_min_shift - pad
                base_z_max = z_max_shift + pad

                # 基準場の z グリッドは本計算の dz に合わせる
                Nzb = int(math.ceil((base_z_max - base_z_min) / dz)) + 1
                base_z_vec = base_z_min + dz * np.arange(Nzb)
                R_base, Z_base = np.meshgrid(self.r_vec, base_z_vec)

                # 3a) z=0 に置いた単一ループ（有限径）を I=1 で離散化
                fils = self._discretize_wire_as_filaments(R0, 0.0, D0, 1.0)

                # 3b) 基準場を一度だけ計算
                Br_base = np.zeros_like(R_base); Bz_base = np.zeros_like(Z_base)
                for f in tqdm(fils, desc=f"fast: base field (R={R0:g}, D={D0:g})"):
                    Br_i, Bz_i = self._B_loop_rz(R_base, Z_base, a=f["a"], z0=f["z0"], I=f["I"])
                    Br_base += Br_i; Bz_base += Bz_i  # I=1.0

                # 3c) 各コイルを z シフトして重ね合わせ（線形性）
                #     z_shifted = self.Z - z_i を base_z_vec に沿って 1次補間（rは完全一致）
                Nr_ = len(self.r_vec); Nz_ = len(self.z_vec)
                J_idx = np.broadcast_to(np.arange(Nr_)[None, :], (Nz_, Nr_))  # 列インデックス

                for z_i, I_i in tqdm(list(zip(z_list, I_list)), desc="fast: shift & add"):
                    Zs = self.Z - z_i  # (Nz,Nr)

                    # インデックス探索（全点同時）：i1 = first index where base_z_vec[i1] >= Zs
                    flat = Zs.ravel()
                    i1 = np.searchsorted(base_z_vec, flat, side="left")
                    i1 = np.clip(i1, 1, len(base_z_vec) - 1)
                    i0 = i1 - 1
                    i1 = i1.reshape(Zs.shape); i0 = i0.reshape(Zs.shape)

                    z0 = base_z_vec[i0]; z1 = base_z_vec[i1]
                    denom = (z1 - z0)
                    t = np.divide(Zs - z0, denom, out=np.zeros_like(Zs), where=denom != 0.0)

                    # 軸0（z方向）での取り出し
                    Br0 = Br_base[i0, J_idx]; Br1 = Br_base[i1, J_idx]
                    Bz0 = Bz_base[i0, J_idx]; Bz1 = Bz_base[i1, J_idx]

                    Br_tot += I_i * ((1.0 - t) * Br0 + t * Br1)
                    Bz_tot += I_i * ((1.0 - t) * Bz0 + t * Bz1)

            self.Br, self.Bz = Br_tot, Bz_tot
            self.Bmag = np.sqrt(Br_tot**2 + Bz_tot**2)

    # ---------- 可視化（従来どおり） ----------
    def plot(self,
             component: str = "Bmag",
             heatmap_colorscale: str = "Viridis",
             show_vectors: bool = True,
             quiver_stride: int = 6,
             quiver_maxlen_frac: float = 0.12,
             quiver_scale: Optional[float] = None,
             title: Optional[str] = None,
             show: bool = True,
             width: Optional[int] = None,
             height: Optional[int] = None,
             zmin: Optional[float] = None,
             zmax: Optional[float] = None
             ) -> go.Figure:
        with self._timer("plot"):
            assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
            assert self.Br is not None and self.Bz is not None, "No field arrays."
            Zplot = self.Bmag if component.lower() != "bz" else self.Bz
            z_title = "|B| [T]" if component.lower() != "bz" else "Bz [T]"

            fig = go.Figure()
            fig.add_trace(go.Heatmap(
                x=self.r_vec, y=self.z_vec, z=Zplot,
                colorscale=heatmap_colorscale,
                colorbar=dict(title=z_title),
                zmin=zmin, zmax=zmax
            ))

            if show_vectors:
                s = max(1, int(quiver_stride))
                r_sub = self.r_vec[::s]; z_sub = self.z_vec[::s]
                R_sub, Z_sub = np.meshgrid(r_sub, z_sub)
                U = self.Br[::s, ::s]; V = self.Bz[::s, ::s]
                if quiver_scale is None:
                    B_sub = np.sqrt(U*U + V*V)
                    Bref = np.percentile(B_sub[B_sub > 0], 99) if np.any(B_sub > 0) else 1.0
                    r_span = float(self.r_vec[-1] - self.r_vec[0])
                    z_span = float(self.z_vec[-1] - self.z_vec[0])
                    base_len = quiver_maxlen_frac * min(r_span, z_span)
                    scale = base_len / (Bref + 1e-30)
                else:
                    scale = float(quiver_scale)
                qfig = ff.create_quiver(R_sub.flatten(), Z_sub.flatten(),
                                        U.flatten(), V.flatten(),
                                        scale=scale, arrow_scale=0.25,
                                        name="B (Br,Bz)", line_width=1)
                for tr in tqdm(qfig.data, desc="plot: add quiver traces"):
                    fig.add_trace(tr)

            # コイル断面輪郭
            shapes = []
            for c in tqdm(self.coil_info, desc="plot: add wire outlines"):
                R0 = float(c["R"]); z0 = float(c["z"]); D = float(c.get("dwire", 0.0))
                if D > 0:
                    shapes.append(dict(
                        type="circle", xref="x", yref="y", layer="above",
                        x0=R0 - D/2, x1=R0 + D/2, y0=z0 - D/2, y1=z0 + D/2,
                        line=dict(width=2, color="black")
                    ))
            fig.update_layout(
                title=title or f"Magnetic field on (r,z) — Heatmap: {z_title}" + (" + vectors" if show_vectors else ""),
                xaxis_title="r [m]", yaxis_title="z [m]",
                template="plotly_white", shapes=shapes,
                width=width, height=height
            )
            fig.update_yaxes(scaleanchor="x", scaleratio=1)
            if show:
                fig.show()
            return fig

    # ---------- 直線プロファイル / インタラクティブ / 保存 / サンプリング（前回どおり） ----------
    # ・・・（前回ご提供の plot_line_profile / interactive_line_profile / save_line_profile /
    #          save_maps_to_excel / sample_field をそのまま残してください）
    # ---------- 直線上プロファイル ----------
    def plot_line_profile(self,
                          a: float,
                          direction: str = "z",
                          quantity: str = "|B|",
                          title: Optional[str] = None,
                          show: bool = True,
                          width: Optional[int] = None,
                          height: Optional[int] = None
                          ) -> go.Figure:
        with self._timer("plot_line_profile"):
            assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
            assert self.Br is not None and self.Bz is not None and self.Bmag is not None, "No field arrays."

            qkey = quantity.lower()
            if qkey == "br":
                Q = self.Br; y_label = "Br [T]"
            elif qkey == "bz":
                Q = self.Bz; y_label = "Bz [T]"
            else:
                Q = self.Bmag; y_label = "|B| [T]"

            direction = direction.lower()
            if direction == "z":
                r_idx = int(np.argmin(np.abs(self.r_vec - a)))
                x = self.z_vec; y = Q[:, r_idx]
                x_label = "z [m]"; used = self.r_vec[r_idx]
                ttl = title or f"Line profile along z at r≈{used:.6g} m ({quantity})"
            elif direction == "r":
                z_idx = int(np.argmin(np.abs(self.z_vec - a)))
                x = self.r_vec; y = Q[z_idx, :]
                x_label = "r [m]"; used = self.z_vec[z_idx]
                ttl = title or f"Line profile along r at z≈{used:.6g} m ({quantity})"
            else:
                raise ValueError("direction must be 'z' or 'r'.")

            fig = go.Figure()
            fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=quantity))
            fig.update_layout(
                title=ttl,
                xaxis_title=x_label,
                yaxis_title=y_label,
                template="plotly_white",
                width=width, height=height
            )
            if show:
                fig.show()
            return fig

    # ADDED 2025-09-22  
    def save_line_profile(self,
                        path: str,
                        a: float,
                        direction: str = "z",
                        quantity: str = "|B|") -> None:
        """
        現在のフィールド配列から、指定直線（r=a または z=a）のラインプロファイルを計算し、
        Excel(.xlsx)に保存する。ファイルダイアログは使わない。
        - direction='z': r=a に最も近い列で z を横軸に保存
        - direction='r': z=a に最も近い行で r を横軸に保存
        - quantity: "|B|" or "Br" or "Bz"
        """
        with self._timer("save_line_profile"):
            import os
            try:
                import pandas as pd
            except Exception:
                raise RuntimeError("pandas が必要です。`pip install pandas openpyxl` を実行してください。")

            assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
            assert self.Br is not None and self.Bz is not None and self.Bmag is not None, "No field arrays."

            qkey = quantity.lower()
            if qkey == "br":
                Q = self.Br; y_label = "Br [T]"
            elif qkey == "bz":
                Q = self.Bz; y_label = "Bz [T]"
            else:
                Q = self.Bmag; y_label = "|B| [T]"

            direction = direction.lower()
            if direction == "z":
                r_idx = int(np.argmin(np.abs(self.r_vec - a)))
                x = self.z_vec; y = Q[:, r_idx]
                x_label = "z [m]"
            elif direction == "r":
                z_idx = int(np.argmin(np.abs(self.z_vec - a)))
                x = self.r_vec; y = Q[z_idx, :]
                x_label = "r [m]"
            else:
                raise ValueError("direction must be 'z' or 'r'.")

            if not path.lower().endswith(".xlsx"):
                path += ".xlsx"
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

            # openpyxlがあれば優先、無ければxlsxwriter
            engine = "openpyxl"
            try:
                import openpyxl  # noqa
            except Exception:
                engine = "xlsxwriter"

            df = pd.DataFrame({x_label: x, y_label: y})
            with pd.ExcelWriter(path, engine=engine) as writer:
                df.to_excel(writer, sheet_name="LineProfile", index=False)

    def save_maps_to_excel(self, path: str) -> None:
        """
        Br, Bz の2シートをもつExcelを出力する。ダイアログは使わない。
        - シートBr/Bzそれぞれで:
            A列の2行目以降に z
            1行目のB列以降に r
            B2セル以降に値行列
        """
        with self._timer("save_maps_to_excel"):
            import os
            try:
                import xlsxwriter
            except Exception:
                raise RuntimeError("`pip install xlsxwriter` を実行してください。")

            assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
            assert self.Br is not None and self.Bz is not None, "No field arrays."

            if not path.lower().endswith(".xlsx"):
                path += ".xlsx"
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

            Nz, Nr = self.Br.shape
            wb = xlsxwriter.Workbook(path)
            try:
                # --- Br ---
                ws = wb.add_worksheet("Br")
                for j, rv in enumerate(self.r_vec):
                    ws.write(0, 1 + j, float(rv))      # 1行目B列以降に r
                for i, zv in enumerate(self.z_vec):
                    ws.write(1 + i, 0, float(zv))      # A列2行目以降に z
                for i in range(Nz):
                    ws.write_row(1 + i, 1, self.Br[i, :].tolist())  # B2以降に行単位で書き込み

                # --- Bz ---
                ws2 = wb.add_worksheet("Bz")
                for j, rv in enumerate(self.r_vec):
                    ws2.write(0, 1 + j, float(rv))
                for i, zv in enumerate(self.z_vec):
                    ws2.write(1 + i, 0, float(zv))
                for i in range(Nz):
                    ws2.write_row(1 + i, 1, self.Bz[i, :].tolist())
            finally:
                wb.close()

    # ################# ADDED 2025-09-22  


    def interactive_line_profile(self,
                                a_init: float = 0.0,
                                direction_init: str = "z",
                                quantity_init: str = "|B|",
                                width: Optional[int] = None,
                                height: Optional[int] = None,
                                profile_save_path: Optional[str] = None,
                                maps_save_path: Optional[str] = None):
        """
        Jupyter上のGUIで a/方向/量 を変更→即時再描画。
        保存ボタンはダイアログを出さず、引数で渡された保存先パスに即保存します。
        - profile_save_path: 現在のラインプロファイルを保存する .xlsx パス
        - maps_save_path   : Br/Bz マップを保存する .xlsx パス
        """
        with self._timer("interactive_line_profile"):
            try:
                import ipywidgets as widgets
                from IPython.display import display, clear_output
            except Exception as e:
                raise RuntimeError("ipywidgets が必要です。`pip install ipywidgets` を実行してください。") from e

            a_box = widgets.FloatText(value=float(a_init), description="a =", step=0.001, layout=widgets.Layout(width="200px"))
            dir_dd = widgets.Dropdown(options=[("r=a の線に沿って (→ r)", "r"),
                                            ("z=a の線に沿って (→ z)", "z")],
                                    value=direction_init, description="方向", layout=widgets.Layout(width="320px"))
            qty_dd = widgets.Dropdown(options=["|B|", "Br", "Bz"],
                                    value=quantity_init, description="量", layout=widgets.Layout(width="200px"))

            btn_save_profile = widgets.Button(description="現在のラインプロファイルをExcelに保存", button_style="primary")
            btn_save_maps    = widgets.Button(description="Br/Bz マップをExcelに保存")

            out_plot = widgets.Output()
            out_msg  = widgets.Output()

            _state = {"x": None, "y": None, "x_label": "", "y_label": "", "title": "",
                    "a": float(a_init), "dir": direction_init, "qty": quantity_init}

            def _update(_=None):
                with out_plot:
                    clear_output(wait=True)
                    q = qty_dd.value
                    d = dir_dd.value
                    a = float(a_box.value)

                    # 量の選択
                    qkey = q.lower()
                    if qkey == "br":
                        Q = self.Br; y_label = "Br [T]"
                    elif qkey == "bz":
                        Q = self.Bz; y_label = "Bz [T]"
                    else:
                        Q = self.Bmag; y_label = "|B| [T]"

                    if d == "z":
                        r_idx = int(np.argmin(np.abs(self.r_vec - a)))
                        x = self.z_vec; y = Q[:, r_idx]
                        x_label = "z [m]"; used = self.r_vec[r_idx]
                        ttl = f"Line profile along z at r≈{used:.6g} m ({q})"
                    else:
                        z_idx = int(np.argmin(np.abs(self.z_vec - a)))
                        x = self.r_vec; y = Q[z_idx, :]
                        x_label = "r [m]"; used = self.z_vec[z_idx]
                        ttl = f"Line profile along r at z≈{used:.6g} m ({q})"

                    _state.update({"x": x, "y": y, "x_label": x_label, "y_label": y_label,
                                "title": ttl, "a": a, "dir": d, "qty": q})

                    fig = go.Figure()
                    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=q))
                    fig.update_layout(title=ttl, xaxis_title=x_label, yaxis_title=y_label,
                                    template="plotly_white", width=width, height=height)
                    fig.show()

            def _save_profile_clicked(_):
                with out_msg:
                    out_msg.clear_output(wait=True)
                    if profile_save_path is None:
                        print("profile_save_path が未指定です。interactive_line_profile(..., profile_save_path='.../line_profile.xlsx') を渡してください。")
                        return
                    try:
                        # 現在表示中の設定（a, dir, qty）で保存
                        self.save_line_profile(profile_save_path, a=_state["a"], direction=_state["dir"], quantity=_state["qty"])
                        print(f"保存しました: {profile_save_path}")
                    except Exception as e:
                        print(f"保存に失敗しました: {e}")

            def _save_maps_clicked(_):
                with out_msg:
                    out_msg.clear_output(wait=True)
                    if maps_save_path is None:
                        print("maps_save_path が未指定です。interactive_line_profile(..., maps_save_path='.../Br_Bz_maps.xlsx') を渡してください。")
                        return
                    try:
                        self.save_maps_to_excel(maps_save_path)
                        print(f"保存しました: {maps_save_path}")
                    except Exception as e:
                        print(f"保存に失敗しました: {e}")

            a_box.observe(_update, names="value")
            dir_dd.observe(_update, names="value")
            qty_dd.observe(_update, names="value")
            btn_save_profile.on_click(_save_profile_clicked)
            btn_save_maps.on_click(_save_maps_clicked)

            _update()

            ui = widgets.VBox([
                widgets.HBox([a_box, dir_dd, qty_dd]),
                out_plot,
                widgets.HBox([btn_save_profile, btn_save_maps]),
                out_msg
            ])
            display(ui)

    # ---------- 任意点サンプリング（Br, Bz を返す） ----------
    def sample_field(self, r: float, z: float, method: str = "bilinear", clip: bool = True) -> Tuple[float, float]:
        """
        指定座標 (r,z) の Br, Bz を返す。
          method: "bilinear"（双一次補間） or "nearest"（最近傍）
          clip  : 範囲外は境界にクリップ（Falseなら範囲外で ValueError）
        """
        assert self.r_vec is not None and self.z_vec is not None, "Call compute() or load() first."
        assert self.Br is not None and self.Bz is not None, "No field arrays."

        r0, r1 = self.r_vec[0], self.r_vec[-1]
        z0, z1 = self.z_vec[0], self.z_vec[-1]
        rr = float(np.clip(r, r0, r1)) if clip else float(r)
        zz = float(np.clip(z, z0, z1)) if clip else float(z)
        if not (r0 <= rr <= r1 and z0 <= zz <= z1):
            raise ValueError(f"(r,z)=({r},{z}) is out of grid range: r∈[{r0},{r1}], z∈[{z0},{z1}]")

        if method.lower() == "nearest":
            i = int(np.argmin(np.abs(self.z_vec - zz)))
            j = int(np.argmin(np.abs(self.r_vec - rr)))
            return float(self.Br[i, j]), float(self.Bz[i, j])

        # bilinear
        i1 = int(np.searchsorted(self.z_vec, zz, side="left"))
        j1 = int(np.searchsorted(self.r_vec, rr, side="left"))
        i0 = max(0, min(i1 - 1, len(self.z_vec) - 2))
        j0 = max(0, min(j1 - 1, len(self.r_vec) - 2))
        zL, zU = self.z_vec[i0], self.z_vec[i0 + 1]
        rL, rU = self.r_vec[j0], self.r_vec[j0 + 1]
        tz = 0.0 if zU == zL else (zz - zL) / (zU - zL)
        tr = 0.0 if rU == rL else (rr - rL) / (rU - rL)

        def _bilin(Q: np.ndarray) -> float:
            q11 = Q[i0, j0]; q12 = Q[i0, j0 + 1]
            q21 = Q[i0 + 1, j0]; q22 = Q[i0 + 1, j0 + 1]
            return float(
                (1 - tz) * ((1 - tr) * q11 + tr * q12) +
                  tz  * ((1 - tr) * q21 + tr * q22)
            )

        return _bilin(self.Br), _bilin(self.Bz)


In [ ]:
# -------------------- 使い方サンプル --------------------
R_inner = 0.10
N_per_layer = 10
N_layer = 5
d_wire = 0.003
I_coil = 1.0
coil_info_list = []
for i in range(N_layer):
    z_i = (i - (N_layer - 1)/2) * (d_wire + 0.001)
    for j in range(N_per_layer):
        R_j = R_inner + j * (d_wire + 0.001)
        coil_info_list.append({"R": R_j, "z": z_i, "dwire": d_wire, "I": I_coil})

lf = AxisymmetricLoopField(
    coil_info=coil_info_list,
    tiles_per_diameter=11,
    pad_ratio_r=0.5, pad_ratio_z=0.5,
    r_floor=1e-12
)

lf.compute(Nr=801, Nz=801, use_shift_accel=True)


In [ ]:
# ヒートマップ（サイズ/カラーバー範囲を指定）
lf.plot(component="Bmag", heatmap_colorscale="Viridis",
        show_vectors=True, quiver_stride=6, quiver_maxlen_frac=0.12,
        quiver_scale=None, title="Class-based |B|",
        width=900, height=700, zmin=None, zmax=None, show=True)

# 2) 任意点の Br, Bz（双一次補間）
Br_val, Bz_val = lf.sample_field(r=0.07, z=0.01, method="bilinear")
print(Br_val, Bz_val)

# 直線上プロファイル（静的）
# lf.plot_line_profile(a=0.05, direction="z", quantity="|B|",
#                       width=800, height=400, show=True)

# 3) ラインプロファイルをコードから保存（ダイアログなし）
lf.save_line_profile(path="./outputs/line_profile_z_at_r=0.05.xlsx",
                     a=0.05, direction="z", quantity="|B|")

# 4) Br/Bz マップをコードから保存（ダイアログなし）
lf.save_maps_to_excel(path="./outputs/Br_Bz_maps.xlsx")

# 5) JupyterのインタラクティブGUI（保存パスを引数で指定）
# import plotly.io as pio; pio.renderers.default = "notebook_connected"
lf.interactive_line_profile(a_init=0.05, direction_init="z", quantity_init="|B|",
                            width=800, height=400)

lf.print_time_summary()
  